In [ ]:
# %%bash
# uv venv --python 3.11
# source .venv/bin/activate
# uv pip install "openmed[hf]"
# uv pip install langchain_groq langchain langchain-core

: 

## Setup and Installation

In [2]:
# Import required libraries
import os
from pprint import pprint
import json

# Set HuggingFace token (if needed)
# os.environ['HF_TOKEN'] = 'your_token_here'

# Import OpenMed PII functions
from openmed import (
    extract_pii,
    deidentify,
    reidentify,
    PIIEntity,
    DeidentificationResult,
)

# Import smart merging utilities
from openmed import (
    merge_entities_with_semantic_units,
    find_semantic_units,
    calculate_dominant_label,
    PII_PATTERNS,
    PIIPattern,
)

# Import batch processing
from openmed import BatchProcessor, BatchItem, process_batch

print("✅ All imports successful!")

✅ All imports successful!


In [3]:
from dotenv import load_dotenv
import os

load_dotenv()

groq_api = os.getenv("GROQ_API")


---
## 1. Basic PII Extraction

Extract PII entities from clinical text.

---
### Test Case: Teacher Annual Performance Report

A complex, realistic annual review document packed with PII across multiple teachers — names, IDs, addresses, DOBs, phones, emails, SSNs, and more.

In [ ]:
import time
from openmed import ModelLoader

MODEL_NAME = 'openmed/OpenMed-PII-SuperClinical-Large-434M-v1'

load_start = time.perf_counter()
loader = ModelLoader()
loader.create_pipeline(MODEL_NAME, task="token-classification", aggregation_strategy="simple")
load_ms = (time.perf_counter() - load_start) * 1000
print(f"Model loaded in {load_ms:.1f}ms — run the next cell for inference.")

teacher_report_text = """
ANNUAL PERFORMANCE EVALUATION REPORT — ACADEMIC YEAR 2023–2024
Greenfield Unified School District
Prepared by: Dr. Margaret L. Harrington, Superintendent
Office Address: 88 District Boulevard, Springfield, IL 62704
Contact: (217) 555-0192 | m.harrington@greenfieldusd.edu

This report documents the formal annual performance evaluations for all certified teaching staff
employed by Greenfield Unified School District during the 2023–2024 academic year. Evaluations
were conducted in accordance with Illinois Administrative Code Part 50 and district Board Policy
5120. All personally identifiable information contained herein is protected under FERPA and the
Illinois School Student Records Act. Unauthorized disclosure is prohibited. Questions regarding
this document should be directed to the Office of Human Resources at (217) 555-0192 or via email
to hr@greenfieldusd.edu. Physical correspondence should be sent to 88 District Boulevard,
Springfield, IL 62704, Attention: HR Records Division.

------------------------------------------------------------------------
TEACHER 1
------------------------------------------------------------------------
Full Name: Prof. James R. Thornton
Employee ID: EMP-20341
Date of Birth: 07/22/1979
Social Security Number: 547-83-2910
Home Address: 14 Maplewood Lane, Apt 3B, Decatur, IL 62521
Personal Phone: (217) 555-3847
Work Email: j.thornton@greenfieldusd.edu
Emergency Contact: Linda Thornton — (217) 555-9201

Subject Taught: Advanced Placement Chemistry (Grades 11–12)
Years of Service: 14 years

Performance Summary:
Prof. James R. Thornton has consistently demonstrated outstanding pedagogical skill throughout the
2023–2024 academic year. His student pass rate in AP Chemistry reached 91%, the highest in the
district. Peer evaluators noted his structured lesson plans and data-driven approach to formative
assessment. He was formally observed on 04/11/2024 by Principal Dr. Roberta N. Fields
(DOB: 11/03/1968, SSN: 312-67-5490, office@fields.greenfield.edu), who rated his performance as
Exemplary. Thornton attended the National Science Teachers Conference in Denver, CO 80202 on
March 14–16, 2024 and presented a well-received workshop on project-based learning in secondary
chemistry. His laboratory redesign proposal, submitted to the district facilities committee, was
approved for a $32,000 renovation grant funded jointly by the state and the Thornton Family
Foundation. He holds an Illinois Teaching Certificate No. IL-TCH-004821 valid through 06/30/2027
and a National Board Certification (NB-CHEM-2019-7734). His direct deposit is linked to account
ending 6641 at Midwest Community Bank, routing number 071923284. Emergency medical contact:
Dr. Patricia Nguyen, (217) 555-0044, p.nguyen@springfieldmed.org, 901 Medical Plaza,
Springfield, IL 62702.

Professional Development Log (Teacher 1):
  - 08/14/2023: District Induction Day, Greenfield Professional Center, Springfield, IL
  - 09/22/2023: Formative Assessment Strategies Workshop, 44 Lake Drive, Decatur, IL 62522
  - 11/10/2023: AP Chemistry Grader Training, College Board Regional Office, Chicago, IL 60604
  - 03/14/2024: NSTA Annual Conference, Colorado Convention Center, Denver, CO 80202
  - 05/03/2024: Illinois Science Teachers Association Symposium, Normal, IL 61761

Personal Notes on File: Thornton submitted a formal accommodation request on 01/08/2024
referencing a medical diagnosis (confidential under ADA). HR correspondence was handled by
benefits coordinator Ms. Rachel Gibson, r.gibson@greenfieldusd.edu, (217) 555-0033. His
FMLA paperwork was processed by third-party administrator OptumHealth, case reference
FMLA-2024-THN-00482, contact 1-866-555-0120. Thornton's professional references on file:
Dr. Alan Marsh, a.marsh@uiuc.edu, (217) 244-1100, 505 S. Goodwin Avenue, Urbana, IL 61801;
and Prof. Carla Stevenson, c.stevenson@siu.edu, (618) 453-2121, Mail Code 4301, Carbondale, IL 62901.


End of Report. Prepared by: Ms. Beverly K. Cross, Records Officer, bev.cross@greenfieldusd.edu,
(217) 555-8844, 88 District Boulevard, Springfield, IL 62704. Date: 06/20/2024.
"""

expected_entities = [
    # ── Header / Superintendent (lines 4-6, 13-15) ──
    ("first_name", "Margaret"), ("last_name", "Harrington"),
    ("street_address", "88 District Boulevard"), ("city", "Springfield"), ("state", "IL"), ("postcode", "62704"),
    ("phone_number", "(217) 555-0192"), ("email", "m.harrington@greenfieldusd.edu"),
    ("email", "hr@greenfieldusd.edu"),

    # ── Teacher 1: James Thornton ──
    ("first_name", "James"), ("last_name", "Thornton"),
    ("date_of_birth", "07/22/1979"), ("ssn", "547-83-2910"),
    ("street_address", "14 Maplewood Lane"), ("city", "Decatur"), ("state", "IL"), ("postcode", "62521"),
    ("phone_number", "(217) 555-3847"), ("email", "j.thornton@greenfieldusd.edu"),
    ("first_name", "Linda"), ("last_name", "Thornton"), ("phone_number", "(217) 555-9201"),
    # Evaluator
    ("first_name", "Roberta"), ("last_name", "Fields"),
    ("date_of_birth", "11/03/1968"), ("ssn", "312-67-5490"), ("email", "office@fields.greenfield.edu"),
    ("city", "Denver"), ("state", "CO"), ("postcode", "80202"),
    ("bank_routing_number", "071923284"),
    # Emergency medical contact
    ("first_name", "Patricia"), ("last_name", "Nguyen"),
    ("phone_number", "(217) 555-0044"), ("email", "p.nguyen@springfieldmed.org"),
    ("street_address", "901 Medical Plaza"), ("city", "Springfield"), ("state", "IL"), ("postcode", "62702"),
    # Prof dev log addresses
    ("street_address", "44 Lake Drive"), ("city", "Decatur"), ("state", "IL"), ("postcode", "62522"),
    ("city", "Chicago"), ("state", "IL"), ("postcode", "60604"),
    # Personal notes: Rachel Gibson, OptumHealth, Alan Marsh, Carla Stevenson
    ("first_name", "Rachel"), ("last_name", "Gibson"),
    ("email", "r.gibson@greenfieldusd.edu"), ("phone_number", "(217) 555-0033"),
    ("phone_number", "1-866-555-0120"),
    ("first_name", "Alan"), ("last_name", "Marsh"),
    ("email", "a.marsh@uiuc.edu"), ("phone_number", "(217) 244-1100"),
    ("street_address", "505 S. Goodwin Avenue"), ("city", "Urbana"), ("state", "IL"), ("postcode", "61801"),
    ("first_name", "Carla"), ("last_name", "Stevenson"),
    ("email", "c.stevenson@siu.edu"), ("phone_number", "(618) 453-2121"),
    ("city", "Carbondale"), ("state", "IL"), ("postcode", "62901")

]

#count expected:
print(len(expected_entities))



Loading weights: 100%|██████████| 392/392 [00:00<00:00, 4318.80it/s]


Model loaded in 3636.6ms — run the next cell for inference.
605


In [8]:
start = time.perf_counter()
result_teacher = extract_pii(
    teacher_report_text,
    model_name=MODEL_NAME,
    confidence_threshold=0.5,
    use_smart_merging=True,
    loader=loader,
)
latency_ms = (time.perf_counter() - start) * 1000

print(f"Found {len(result_teacher.entities)} PII entities (latency: {latency_ms:.1f}ms):\n")
for i, entity in enumerate(result_teacher.entities, 1):
    print(f"{i:2d}. [{entity.label:25s}] '{entity.text:30s}' (confidence: {entity.confidence:.3f})")

print("\n" + "=" * 80)

Found 1610 PII entities (latency: 127435.3ms):

 1. [date                     ] '2024                          ' (confidence: 0.521)
 2. [first_name               ] 'Margaret                      ' (confidence: 0.997)
 3. [last_name                ] 'Harrington                    ' (confidence: 0.993)
 4. [occupation               ] 'Superintendent                ' (confidence: 0.798)
 5. [street_address           ] '88 District Boulevard         ' (confidence: 0.960)
 6. [city                     ] 'Springfield                   ' (confidence: 0.847)
 7. [state                    ] 'IL                            ' (confidence: 0.834)
 8. [postcode                 ] '62704                         ' (confidence: 0.722)
 9. [phone_number             ] ') 555-0192                    ' (confidence: 0.676)
10. [email                    ] 'm.harrington@greenfieldusd.edu' (confidence: 0.871)
11. [date                     ] '2023–2024                     ' (confidence: 0.889)
12. [state       

In [9]:
# ── Ground truth for the teacher annual report ──────────────────────────────
DOB_AS_DATE_ACCEPTABLE = True


def normalize(text):
    return text.strip().lower().replace("\n", " ").replace("  ", " ")

def labels_match(exp_l, mod_l):
    return exp_l == mod_l or (DOB_AS_DATE_ACCEPTABLE and exp_l == "date_of_birth" and mod_l == "date")

def texts_match(exp_t, mod_t):
    e, m = normalize(exp_t), normalize(mod_t)
    return e == m or e in m or m in e

model_pool = [(e.label, e.text) for e in result_teacher.entities]
matched, missed = [], []
HIGH_VALUE = {"ssn", "date_of_birth", "phone_number", "email", "street_address"}

for exp_l, exp_t in expected_entities:
    found = False
    for i, (m_l, m_t) in enumerate(model_pool):
        if labels_match(exp_l, m_l) and texts_match(exp_t, m_t):
            matched.append((exp_l, exp_t, m_l, m_t))
            model_pool.pop(i)
            found = True
            break
    if not found: missed.append((exp_l, exp_t))

BENIGN_FP = {"date", "education_level", "url", "company_name", "country", "pin", "bank_routing_number"}
fp_crit = [(l, t) for l, t in model_pool if l not in BENIGN_FP]
fp_benign = [(l, t) for l, t in model_pool if l in BENIGN_FP]

tp, fn, fp = len(matched), len(missed), len(model_pool)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0

W = 80
print("=" * W)
print("  DETAILED MODEL PERFORMANCE REPORT")
print("=" * W)
print(f"  Actual Latency   : {latency_ms:,.1f} ms")
print(f"  True Positives   : {tp} (Found matches)")
print(f"  False Positives  : {fp} (Extras detected)")
print(f"  False Negatives  : {fn} (Missed from ground truth)")
print("-" * W)
print(f"  Precision        : {precision:.2%} [TP / (TP + FP)]")
print(f"  Recall           : {recall:.2%} [TP / (TP + FN)]")
print(f"  F1 Score         : {f1:.2%}")

print(f"\n❌ MISSED ENTITIES ({fn})")
for l, t in missed:
    hv = " ☑ HIGH VALUE" if l in HIGH_VALUE else ""
    print(f"   [{l:15s}] '{t}'{hv}")

print(f"\n⚠️ CRITICAL FALSE POSITIVES ({len(fp_crit)}) - Directly impacts Precision")
for l, t in fp_crit:
    print(f"   [{l:15s}] '{t.strip()}'")

if fp_benign:
    print(f"\nℹ️ BENIGN FALSE POSITIVES ({len(fp_benign)}) - Extra contextual dates/info")
    for l, t in fp_benign[:10]:
        print(f"   [{l:15s}] '{t.strip()}'")
    if len(fp_benign) > 10: print("   ... (truncated)")

print("=" * W)

  DETAILED MODEL PERFORMANCE REPORT
  Actual Latency   : 127,435.3 ms
  True Positives   : 592 (Found matches)
  False Positives  : 1018 (Extras detected)
  False Negatives  : 13 (Missed from ground truth)
--------------------------------------------------------------------------------
  Precision        : 36.77% [TP / (TP + FP)]
  Recall           : 97.85% [TP / (TP + FN)]
  F1 Score         : 53.45%

❌ MISSED ENTITIES (13)
   [phone_number   ] '(618) 453-2121' ☑ HIGH VALUE
   [phone_number   ] '1-800-555-0220' ☑ HIGH VALUE
   [phone_number   ] '(312) 555-6610' ☑ HIGH VALUE
   [phone_number   ] '(800) 555-0180' ☑ HIGH VALUE
   [state          ] 'DC'
   [phone_number   ] '(202) 555-0190' ☑ HIGH VALUE
   [state          ] 'DC'
   [phone_number   ] '1-866-555-0120' ☑ HIGH VALUE
   [phone_number   ] '1-888-555-0100' ☑ HIGH VALUE
   [phone_number   ] '1-800-555-0294' ☑ HIGH VALUE
   [phone_number   ] '(217) 555-0033' ☑ HIGH VALUE
   [phone_number   ] '(217) 555-0010' ☑ HIGH VALUE
   [phone

In [74]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field
from typing import List
import json
import time
import re

class PIIEntitySchema(BaseModel):
    label: str = Field(description="PII category")
    text: str = Field(description="PII text")

class PIIExtraction(BaseModel):
    entities: List[PIIEntitySchema]

parser = PydanticOutputParser(pydantic_object=PIIExtraction)
# Switching to a more stable model for reliable output
llm = ChatGroq(model="llama-3.3-70b-versatile", api_key=groq_api)

prompt_template = """Task: Extract PII from the text for each entities. no extra sentence, only strctured output.
{format_instructions}
Valid labels: first_name, last_name, occupation, street_address, city, state, postcode, phone_number, email, date_of_birth, ssn, date.
Text: {text}"""

formatted_prompt = prompt_template.format(format_instructions=parser.get_format_instructions(), text=teacher_report_text)

start_llm = time.perf_counter()
llm_response = llm.invoke(formatted_prompt)
print(llm_response)
llm_latency_ms = (time.perf_counter() - start_llm) * 1000

try:
    content = llm_response.content.strip()
    # Simple logic to handle markdown blocks if model adds them
    if '```' in content:
        content = re.search(r'```(?:json)?(.*?)```', content, re.DOTALL).group(1).strip()

    parsed_output = parser.parse(content)

    class MockEntity:
        def __init__(self, label, text):
            self.label = label
            self.text = text
            self.confidence = 1.0

    llm_entities = [MockEntity(e.label, e.text) for e in parsed_output.entities]
    print(f"LLM successfully found {len(llm_entities)} entities (latency: {llm_latency_ms:.1f}ms)")
except Exception as e:
    print(f"Parsing Error: {e}")
    print("Raw content snippet:", llm_response.content[-100:] if llm_response.content else 'EMPTY')
    llm_entities = []

content='```\n{\n  "entities": [\n    {\n      "label": "first_name",\n      "text": "Margaret"\n    },\n    {\n      "label": "last_name",\n      "text": "Harrington"\n    },\n    {\n      "label": "street_address",\n      "text": "88 District Boulevard"\n    },\n    {\n      "label": "city",\n      "text": "Springfield"\n    },\n    {\n      "label": "state",\n      "text": "IL"\n    },\n    {\n      "label": "postcode",\n      "text": "62704"\n    },\n    {\n      "label": "phone_number",\n      "text": "(217) 555-0192"\n    },\n    {\n      "label": "email",\n      "text": "m.harrington@greenfieldusd.edu"\n    },\n    {\n      "label": "first_name",\n      "text": "James"\n    },\n    {\n      "label": "last_name",\n      "text": "Thornton"\n    },\n    {\n      "label": "date_of_birth",\n      "text": "07/22/1979"\n    },\n    {\n      "label": "ssn",\n      "text": "547-83-2910"\n    },\n    {\n      "label": "street_address",\n      "text": "14 Maplewood Lane"\n    },\n    {\n  

In [78]:
print(f"--- LLM DETECTED ENTITIES (Total: {len(llm_entities)}) ---\n")
for i, entity in enumerate(llm_entities, 1):
    print(f"{i:2d}. [{entity.label:20s}] '{entity.text}'")

--- LLM DETECTED ENTITIES (Total: 77) ---

 1. [first_name          ] 'Margaret'
 2. [last_name           ] 'Harrington'
 3. [street_address      ] '88 District Boulevard'
 4. [city                ] 'Springfield'
 5. [state               ] 'IL'
 6. [postcode            ] '62704'
 7. [phone_number        ] '(217) 555-0192'
 8. [email               ] 'm.harrington@greenfieldusd.edu'
 9. [first_name          ] 'James'
10. [last_name           ] 'Thornton'
11. [date_of_birth       ] '07/22/1979'
12. [ssn                 ] '547-83-2910'
13. [street_address      ] '14 Maplewood Lane'
14. [city                ] 'Decatur'
15. [state               ] 'IL'
16. [postcode            ] '62521'
17. [phone_number        ] '(217) 555-3847'
18. [email               ] 'j.thornton@greenfieldusd.edu'
19. [first_name          ] 'Linda'
20. [last_name           ] 'Thornton'
21. [phone_number        ] '(217) 555-9201'
22. [first_name          ] 'Roberta'
23. [last_name           ] 'Fields'
24. [date_of_birth 

In [79]:
# 1. Calculate LLM Metrics
llm_model_pool = [(e.label, e.text) for e in llm_entities]
llm_matched, llm_missed = [], []

for exp_l, exp_t in expected_entities:
    found = False
    for i, (m_l, m_t) in enumerate(llm_model_pool):
        if labels_match(exp_l, m_l) and texts_match(exp_t, m_t):
            llm_matched.append((exp_l, exp_t, m_l, m_t))
            llm_model_pool.pop(i)
            found = True
            break
    if not found: llm_missed.append((exp_l, exp_t))

llm_tp, llm_fn, llm_fp = len(llm_matched), len(llm_missed), len(llm_model_pool)
llm_precision = llm_tp / (llm_tp + llm_fp) if (llm_tp + llm_fp) > 0 else 0
llm_recall = llm_tp / (llm_tp + llm_fn) if (llm_tp + llm_fn) > 0 else 0
llm_f1 = 2*llm_precision*llm_recall/(llm_precision+llm_recall) if (llm_precision+llm_recall) > 0 else 0

# 2. Display the Benchmark Table
print("=" * 80)
print("  BENCHMARK COMPARISON: OPENMED vs LLM (Llama-3.3-70B)")
print("=" * 80)
print(f"{'Metric':<15} | {'OpenMed (Local)':<18} | {'LLM (Groq API)':<18}")
print("-" * 80)
print(f"{'Precision':<15} | {precision:<18.2%} | {llm_precision:<18.2%}")
print(f"{'Recall':<15} | {recall:<18.2%} | {llm_recall:<18.2%}")
print(f"{'F1 Score':<15} | {f1:<18.2%} | {llm_f1:<18.2%}")
print(f"{'Latency':<15} | {latency_ms:<15.1f} ms | {llm_latency_ms:<15.1f} ms")
print("=" * 80)

print(f"\nEntities detected: OpenMed (122) vs LLM ({len(llm_entities)})")

  BENCHMARK COMPARISON: OPENMED vs LLM (Llama-3.3-70B)
Metric          | OpenMed (Local)    | LLM (Groq API)    
--------------------------------------------------------------------------------
Precision       | 61.48%             | 94.81%            
Recall          | 94.94%             | 92.41%            
F1 Score        | 74.63%             | 93.59%            
Latency         | 43865.1         ms | 2850.8          ms

Entities detected: OpenMed (122) vs LLM (77)


In [10]:
print("=" * 80)
print("  MISSED ENTITIES COMPARISON: OPENMED vs LLM")
print("=" * 80)

# OpenMed Missed
om_missed_list = [f"[{l:15s}] '{t}'" for l, t in missed]
# LLM Missed
llm_missed_list = [f"[{l:15s}] '{t}'" for l, t in llm_missed]

print(f"OpenMed Missed ({len(om_missed_list)}):")
for m in om_missed_list:
    print(f"  ❌ {m}")

print(f"\nLLM Missed ({len(llm_missed_list)}):")
for m in llm_missed_list:
    print(f"  ❌ {m}")

# Find common misses
common_misses = [m for m in om_missed_list if m in llm_missed_list]
if common_misses:
    print(f"\nCommonly Missed by Both ({len(common_misses)}):")
    for m in common_misses:
        print(f"  ⚠️ {m}")
else:
    print("\nNo overlapping misses found.")

  MISSED ENTITIES COMPARISON: OPENMED vs LLM


NameError: name 'llm_missed' is not defined

---
## 3. De-identification Methods

OpenMed supports multiple de-identification methods to protect patient privacy.

In [65]:
# Clinical note for de-identification
patient_note = """
CLINICAL NOTE
=============
Patient Name: Dr. Sarah Johnson
Date of Birth: 03/15/1975
MRN: 87654321
Social Security: 123-45-6789
Contact: (555) 987-6543
Email: sarah.j@hospital.org
Address: 456 Oak Avenue, Boston, MA 02115

Admission Date: 12/20/2024
Discharge Date: 12/25/2024

DIAGNOSIS: Type 2 Diabetes Mellitus
"""

print("Original Clinical Note:")
print(patient_note)
print("\n" + "=" * 80)

Original Clinical Note:

CLINICAL NOTE
Patient Name: Dr. Sarah Johnson
Date of Birth: 03/15/1975
MRN: 87654321
Social Security: 123-45-6789
Contact: (555) 987-6543
Email: sarah.j@hospital.org
Address: 456 Oak Avenue, Boston, MA 02115

Admission Date: 12/20/2024
Discharge Date: 12/25/2024

DIAGNOSIS: Type 2 Diabetes Mellitus




### Method 1: Mask (Placeholder replacement)

In [9]:
print("=" * 80)
print("METHOD 1: MASK (Placeholder replacement)")
print("=" * 80)

result_mask = deidentify(
    patient_note,
    method="mask",
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.6,
    use_smart_merging=True,
)

print("De-identified text:")
print(result_mask.deidentified_text)

# ✅ The library returns `pii_entities` (not `entities`)
entities = getattr(result_mask, "pii_entities", None) or getattr(result_mask, "entities", [])

print(f"\nEntities masked: {len(entities)}")

# Optional: sort by position for nicer display
entities = sorted(entities, key=lambda e: getattr(e, "start", 0))

for entity in entities[:5]:  # Show first 5
    label = getattr(entity, "label", getattr(entity, "entity_type", "UNKNOWN"))
    text = getattr(entity, "text", "")
    redacted = getattr(entity, "redacted_text", "")
    conf = getattr(entity, "confidence", None)
    span = (getattr(entity, "start", None), getattr(entity, "end", None))

    conf_str = f"{conf:.3f}" if isinstance(conf, (int, float)) else "n/a"
    print(f"  [{label}] '{text}' -> '{redacted}' conf={conf_str} span={span}")

METHOD 1: MASK (Placeholder replacement)


Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

De-identified text:
CLINICAL NOTE
Patient Name: Dr. [first_name] [last_name]
Date of Birth: [date_of_birth]
MRN: 87654321
Social Security: [ssn]
Contact: ([phone_number]
Email: [email]
Address: [street_address], [city], [state] [postcode]

Admission Date: [date]
Discharge Date: [date]

DIAGNOSIS: Type 2 Diabetes Mellitus

Entities masked: 12
  [first_name] 'Sarah' -> '[first_name]' conf=1.000 span=(46, 51)
  [last_name] 'Johnson' -> '[last_name]' conf=0.999 span=(52, 59)
  [date_of_birth] '03/15/1975' -> '[date_of_birth]' conf=0.849 span=(75, 85)
  [ssn] '123-45-6789' -> '[ssn]' conf=0.926 span=(117, 128)
  [phone_number] '555) 987-6543' -> '[phone_number]' conf=0.659 span=(139, 152)


### Method 2: Remove (Complete removal)

In [ ]:
print("=" * 80)
print("METHOD 2: REMOVE (Complete removal)")
print("=" * 80)

result_remove = deidentify(
    patient_note,
    method="remove",
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.6,
    use_smart_merging=True,
)

print("De-identified text:")
print(result_remove.deidentified_text)

# ✅ Use `pii_entities` (fallback included for robustness)
entities = getattr(result_remove, "pii_entities", None) or getattr(result_remove, "entities", [])

print(f"\nEntities removed: {len(entities)}")

### Method 3: Replace (Synthetic data)

In [ ]:
print("=" * 80)
print("METHOD 3: REPLACE (Synthetic data)")
print("=" * 80)

result_replace = deidentify(
    patient_note,
    method="replace",
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.6,
    use_smart_merging=True,
)

print("De-identified text:")
print(result_replace.deidentified_text)

# ✅ `DeidentificationResult` uses `pii_entities`
entities = getattr(result_replace, "pii_entities", None) or getattr(result_replace, "entities", [])

print(f"\nEntities replaced: {len(entities)}")

for entity in sorted(entities, key=lambda e: getattr(e, "start", 0))[:5]:
    label = getattr(entity, "label", getattr(entity, "entity_type", "UNKNOWN"))
    text = getattr(entity, "text", "")
    # for replace/mask, this often holds the replacement value or placeholder
    repl = getattr(entity, "redacted_text", "")
    print(f"  [{label}] '{text}' -> '{repl}'")

### Method 4: Hash (Cryptographic hashing)

In [ ]:
print("=" * 80)
print("METHOD 4: HASH (Cryptographic hashing)")
print("=" * 80)

result_hash = deidentify(
    patient_note,
    method="hash",
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.6,
    use_smart_merging=True,
)

print("De-identified text (first 500 chars):")
text = result_hash.deidentified_text or ""
print((text[:500] + "...") if len(text) > 500 else text)

# ✅ Use `pii_entities` (fallback included)
entities = getattr(result_hash, "pii_entities", None) or getattr(result_hash, "entities", [])

print(f"\nEntities hashed: {len(entities)}")
print("\nExample hashed values:")

for entity in sorted(entities, key=lambda e: getattr(e, "start", 0))[:3]:
    label = getattr(entity, "label", getattr(entity, "entity_type", "UNKNOWN"))
    original = getattr(entity, "text", "")
    hashed = getattr(entity, "hash_value", None) or getattr(entity, "redacted_text", "")
    print(f"  [{label}] Original: '{original}'  Hashed: '{hashed}'")

### Method 5: Shift Dates (Date shifting)

In [ ]:
print("=" * 80)
print("METHOD 5: SHIFT_DATES (Preserves temporal relationships)")
print("=" * 80)

result_shift = deidentify(
    patient_note,
    method="shift_dates",
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.6,
    use_smart_merging=True,
    date_shift_days=365,  # Shift by 1 year
)

print("De-identified text:")
print(result_shift.deidentified_text)

# ✅ Use `pii_entities` (fallback included)
entities = getattr(result_shift, "pii_entities", None) or getattr(result_shift, "entities", [])

date_entities = [
    e for e in entities
    if "date" in getattr(e, "label", getattr(e, "entity_type", "")).lower()
]

print("\nDate entities shifted:")
for e in sorted(date_entities, key=lambda x: getattr(x, "start", 0)):
    label = getattr(e, "label", getattr(e, "entity_type", "UNKNOWN"))
    original = getattr(e, "text", "")
    shifted = getattr(e, "redacted_text", "")  # often holds the shifted date or replacement
    print(f"  [{label}] '{original}' -> '{shifted}'")

print("Note: Temporal relationships between dates are preserved!")

---
## 4. Re-identification

Reverse de-identification using stored mappings.

In [ ]:
print("=" * 80)
print("RE-IDENTIFICATION")
print("=" * 80)

# De-identify with mapping
print("Step 1: De-identify with keep_mapping=True")
result_with_mapping = deidentify(
    patient_note,
    method="mask",
    model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
    confidence_threshold=0.6,
    keep_mapping=True,  # Keep mapping for re-identification
    use_smart_merging=True
)

print(f"\nDe-identified text (first 200 chars):")
print(result_with_mapping.deidentified_text[:200] + "...")

print(f"\nMapping created: {len(result_with_mapping.mapping)} entries")
print("\nFirst 5 mapping entries:")
for i, (redacted, original) in enumerate(list(result_with_mapping.mapping.items())[:5], 1):
    print(f"  {i}. '{redacted}' → '{original}'")

# Re-identify
print("\n" + "-" * 80)
print("Step 2: Re-identify using the mapping")
original_text = reidentify(
    result_with_mapping.deidentified_text,
    result_with_mapping.mapping
)

print(f"\nRe-identified text (first 300 chars):")
print(original_text[:300] + "...")

# Verify
print("\n" + "-" * 80)
print("Verification:")
original_clean = patient_note.strip()
reidentified_clean = original_text.strip()
if original_clean == reidentified_clean:
    print("✅ SUCCESS: Re-identification is perfect!")
else:
    print(f"⚠️  Difference detected (usually whitespace/formatting)")
    print(f"   Original length: {len(original_clean)}")
    print(f"   Re-identified length: {len(reidentified_clean)}")

---
## 5. Batch Processing

Efficiently process multiple clinical notes.

In [ ]:
from openmed import BatchProcessor

print("=" * 80)
print("BATCH PROCESSING")
print("=" * 80)

batch_texts = [
    "Patient: John Doe, DOB: 01/15/1970, SSN: 123-45-6789",
    "Dr. Sarah Johnson, Phone: (555) 123-4567, Email: sarah@email.com",
    "MRN: 87654321, Admission: 2024-03-20, Discharge: 2024-03-25",
    "Address: 123 Main Street, Boston, MA 02101, ZIP: 02101",
    "Contact: patient.name@hospital.org, Emergency: (555) 987-6543",
]

processor = BatchProcessor(
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    confidence_threshold=0.5,
    group_entities=True,
    continue_on_error=True,
    # IMPORTANT: do NOT pass use_smart_merging here if your installed version triggers the HF pipeline error
)

batch_result = processor.process_texts(batch_texts)

print("Batch processing completed!")
print(f"  Total items: {batch_result.total_items}")
print(f"  Successful: {batch_result.successful_items}")
print(f"  Failed: {batch_result.failed_items}")
print(f"  Total processing time: {batch_result.total_processing_time:.2f}s")

print("\n" + "-" * 80)
print("Results per note:\n")

for item_result in batch_result.items:
    if not item_result.success:
        print(f"❌ {item_result.id}: {item_result.error}")
        continue

    # In BatchProcessor results, entities usually live under item_result.result.entities
    ents = item_result.result.entities
    print(f"📄 {item_result.id}:")
    print(f"   Entities found: {len(ents)}")
    for entity in ents[:3]:
        print(f"     - [{entity.label}] '{entity.text}'")
    if len(ents) > 3:
        print(f"     ... and {len(ents) - 3} more")
    print()

### Batch De-identification

In [ ]:
print("=" * 80)
print("BATCH DE-IDENTIFICATION (extract in batch, then deidentify)")
print("=" * 80)

ids = [f"note_{i+1}" for i in range(len(batch_texts))]

# 1) Batch extraction (no use_smart_merging here in YOUR install; it breaks HF pipeline creation)
batch_result = process_batch(
    batch_texts,
    model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
    ids=ids,
    confidence_threshold=0.6,
    batch_size=2,
)

print(f"Batch extraction completed!")
print(f"  Successful: {batch_result.successful_items}/{batch_result.total_items}\n")

# 2) Deidentify each text (this API supports use_smart_merging in your environment)
print("De-identified texts:\n")

for item in batch_result.items:
    item_id = getattr(item, "id", None) or getattr(item, "item_id", "unknown")

    if not getattr(item, "success", False):
        err = getattr(item, "error", None) or getattr(item, "exception", None)
        print(f"❌ {item_id} failed during extraction: {err}")
        continue

    # run deidentify for the same text
    original_text = item.text if hasattr(item, "text") else batch_texts[ids.index(item_id)]
    deid = deidentify(
        original_text,
        method="mask",
        model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
        confidence_threshold=0.6,
        use_smart_merging=True,
    )

    print(f"📄 {item_id}:")
    print(deid.deidentified_text)
    print()

---
## 6. Confidence Thresholding

Control precision vs recall trade-off.

In [ ]:
print("=" * 80)
print("CONFIDENCE THRESHOLDING")
print("=" * 80)

test_text = "Patient: Jane Doe, DOB: 05/20/1985, Phone: 555-1234, Email: jane@email.com"
print(f"Input: {test_text}\n")

thresholds = [0.3, 0.5, 0.7, 0.9]

for threshold in thresholds:
    result = extract_pii(
        test_text,
        model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
        confidence_threshold=threshold,
        use_smart_merging=True
    )

    print(f"Threshold: {threshold:.1f} → {len(result.entities)} entities")
    for entity in result.entities:
        print(f"  [{entity.label:20s}] '{entity.text:25s}' (conf: {entity.confidence:.3f})")
    print()

print("-" * 80)
print("Guidelines:")
print("  • threshold=0.3-0.5: High recall (catch more PII, more false positives)")
print("  • threshold=0.5-0.7: Balanced (RECOMMENDED for most use cases)")
print("  • threshold=0.7-0.9: High precision (fewer false positives, may miss some PII)")

---
## 7. Custom PII Patterns

Add domain-specific patterns for your organization.

In [ ]:
print("=" * 80)
print("CUSTOM PII PATTERNS")
print("=" * 80)

# Define custom patterns
custom_patterns = [
    PIIPattern(
        pattern=r'\bEMP-\d{6}\b',  # Employee ID format: EMP-123456
        entity_type='employee_id',
        priority=10
    ),
    PIIPattern(
        pattern=r'\bPID-\d{8}\b',  # Patient ID format: PID-12345678
        entity_type='patient_id',
        priority=9
    ),
    PIIPattern(
        pattern=r'\b[A-Z]{2}-\d{4}-[A-Z]\b',  # Custom format: AB-1234-X
        entity_type='internal_code',
        priority=8
    ),
]

print(f"Defined {len(custom_patterns)} custom patterns:\n")
for p in custom_patterns:
    print(f"  [{p.entity_type:20s}] Priority: {p.priority}, Pattern: {p.pattern}")

# Test text with custom identifiers
custom_text = """
Employee: EMP-123456
Patient ID: PID-87654321
Department Code: HR-2024-A
Regular SSN: 123-45-6789
"""

print("\n" + "-" * 80)
print("Test text:")
print(custom_text)

# Find custom semantic units
print("-" * 80)
print("Detected units with custom patterns:\n")

# First get model predictions
result = extract_pii(
    custom_text,
    model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
    confidence_threshold=0.5,
    use_smart_merging=False  # Get raw predictions first
)

# Convert to dict format
entity_dicts = [
    {
        'entity_type': e.label,
        'score': e.confidence,
        'start': e.start,
        'end': e.end,
        'word': e.text
    }
    for e in result.entities
]

# Merge with custom patterns
merged = merge_entities_with_semantic_units(
    entity_dicts,
    result.text,
    patterns=custom_patterns,  # Add custom patterns
    use_semantic_patterns=True,
    prefer_model_labels=False  # Prefer pattern labels for custom types
)

print(f"Found {len(merged)} entities (including custom types):\n")
for entity in merged:
    label = entity['entity_type']
    text = entity['word']
    conf = entity['score']
    is_custom = label in ['employee_id', 'patient_id', 'internal_code']
    marker = "🆕" if is_custom else "  "
    print(f"{marker} [{label:20s}] '{text}' (confidence: {conf:.3f})")

print("\n🆕 = Custom pattern detected")

---
## 8. Clinical Use Cases

Real-world clinical scenarios.

### Use Case 1: Discharge Summary

In [ ]:
discharge_summary = """
DISCHARGE SUMMARY
=====================================
Patient Name: Michael Anderson
MRN: 98765432
Date of Birth: 08/12/1968
Admission Date: 01/05/2025
Discharge Date: 01/10/2025
Attending Physician: Dr. Emily Carter

PRIMARY DIAGNOSIS:
Acute myocardial infarction

HOSPITAL COURSE:
Mr. Anderson is a 56-year-old male who presented to the emergency
department on 01/05/2025 with chest pain. He was admitted for
cardiac catheterization and intervention.

CONTACT INFORMATION:
Phone: (555) 234-5678
Email: m.anderson@email.com
Emergency Contact: Jane Anderson (Wife) - (555) 234-5679

FOLLOW-UP:
Patient scheduled for follow-up on 01/24/2025 at the cardiology clinic.
"""

print("=" * 80)
print("USE CASE 1: Discharge Summary De-identification")
print("=" * 80)

# De-identify for research database
deid_discharge = deidentify(
    discharge_summary,
    method="mask",
    model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
    confidence_threshold=0.6,
    use_smart_merging=True
)

print("De-identified Discharge Summary:")
print(deid_discharge.deidentified_text)

print("\n" + "-" * 80)
print(f"PII entities protected: {len(deid_discharge.pii_entities)}")
# Check for adjacent placeholders (quality check)
if '][' in deid_discharge.deidentified_text:
    print("❌ Adjacent placeholders detected - fragmentation issue!")
else:
    print("✅ Clean de-identification - no adjacent placeholders!")

### Use Case 2: Research Dataset Preparation

In [ ]:
research_notes = [
    "Patient 001: John Smith, DOB 03/15/1975, diagnosed with T2DM on 12/20/2024",
    "Patient 002: Sarah Johnson, DOB 08/22/1982, A1C 8.5%, started metformin 01/05/2025",
    "Patient 003: Robert Williams, DOB 11/30/1965, BMI 32.1, blood pressure 145/90",
]

print("=" * 80)
print("USE CASE 2: Research Dataset Preparation")
print("=" * 80)
print(f"Processing {len(research_notes)} patient notes for research...\n")

# For batch de-identification, use deidentify() on each text
# BatchProcessor.process_items() is for extraction only

print("De-identified research dataset (date shifting by 180 days):\n")

for i, note in enumerate(research_notes, 1):
    patient_id = f"patient_{i:03d}"

    # De-identify each note with date shifting
    deid_result = deidentify(
        note,
        method="shift_dates",
        model_name="openmed/OpenMed-PII-SuperClinical-Large-434M-v1",
        confidence_threshold=0.6,
        use_smart_merging=True,
        date_shift_days=180,
        keep_mapping=True,
    )

    print(f"{patient_id}: {deid_result.deidentified_text}")

print("\n" + "-" * 80)
print("✅ Research dataset ready!")
print("   - All dates shifted by 180 days")
print("   - Temporal relationships preserved")
print("   - Audit mapping available for IRB review")

### Use Case 3: HIPAA Compliance Audit

In [ ]:
print("=" * 80)
print("USE CASE 3: HIPAA Compliance Audit")
print("=" * 80)

# HIPAA Safe Harbor requires removal of 18 identifiers
hipaa_text = """
Patient: Jane Doe
DOB: 05/15/1980
SSN: 987-65-4321
Address: 789 Pine Street, Unit 4B, Seattle, WA 98101
Phone: (206) 555-1234
Fax: (206) 555-1235
Email: jane.doe@email.com
Medical Record: MRN-12345678
Account Number: ACCT-987654
Device ID: DEVICE-ABC123
IP Address: 192.168.1.100
Vehicle: License plate ABC-1234
Biometric: Fingerprint on file
Photo: Patient photo available
URL: https://patient-portal.hospital.com/jane-doe
"""

# Extract all PII for audit
hipaa_result = extract_pii(
    hipaa_text,
    model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
    confidence_threshold=0.5,
    use_smart_merging=True
)

print(f"HIPAA Compliance Check:\n")
print(f"Total PII entities detected: {len(hipaa_result.entities)}\n")

# Group by type
from collections import Counter
pii_types = Counter(e.label for e in hipaa_result.entities)

print("PII Categories Found:")
for pii_type, count in sorted(pii_types.items()):
    print(f"  • {pii_type}: {count} instance(s)")

# HIPAA 18 identifiers checklist
hipaa_18_identifiers = [
    'names', 'geographic', 'dates', 'phone', 'fax', 'email', 'ssn',
    'medical_record', 'account_number', 'certificate', 'vehicle',
    'device', 'url', 'ip_address', 'biometric', 'photo', 'unique_id'
]

print("\n" + "-" * 80)
print("HIPAA Safe Harbor Compliance:")
detected_types = set(e.label.lower() for e in hipaa_result.entities)
covered = sum(1 for identifier in hipaa_18_identifiers
              if any(identifier in dt for dt in detected_types))

print(f"  Coverage: {covered}/{len(hipaa_18_identifiers)} HIPAA identifier categories")
print("  ✅ Ready for HIPAA-compliant de-identification")

---
## 9. Visualization

Display results with highlighting.

In [ ]:
from IPython.display import HTML, display

def highlight_entities(text, entities):
    """Create HTML with highlighted PII entities."""
    # Define colors for different entity types
    colors = {
        'name': '#FFB3BA',
        'first_name': '#FFB3BA',
        'last_name': '#FFB3BA',
        'date': '#BAFFC9',
        'date_of_birth': '#BAFFC9',
        'ssn': '#BAE1FF',
        'phone': '#FFFFBA',
        'email': '#FFDFBA',
        'address': '#E0BBE4',
        'medical_record_number': '#D4F1F4',
    }

    # Sort entities by start position (reverse for replacement)
    sorted_entities = sorted(entities, key=lambda e: e.start, reverse=True)

    highlighted = text
    for entity in sorted_entities:
        color = colors.get(entity.label.lower(), '#E8E8E8')
        replacement = (
            f'<span style="background-color: {color}; padding: 1px 2px; '
            f'border-radius: 3px; font-weight: bold;" '
            f'title="{entity.label} (confidence: {entity.confidence:.2f})">{entity.text}</span>'
        )
        highlighted = (
            highlighted[:entity.start] + replacement + highlighted[entity.end:]
        )

    return f'<div style="font-family: monospace; white-space: pre-wrap; padding: 10px; background: #f8f8f8; border-radius: 5px;">{highlighted}</div>'

# Example visualization
viz_text = """Patient: Dr. Sarah Johnson
DOB: 03/15/1975
SSN: 123-45-6789
Phone: (555) 123-4567
Email: sarah.j@email.com
Address: 456 Oak Ave, Boston, MA 02115"""

viz_result = extract_pii(
    viz_text,
    model_name='openmed/OpenMed-PII-SuperClinical-Large-434M-v1',
    confidence_threshold=0.5,
    use_smart_merging=True
)

print("=" * 80)
print("VISUALIZATION: Highlighted PII Entities")
print("=" * 80)
print("\nHover over highlighted text to see entity type and confidence.\n")

html = highlight_entities(viz_result.text, viz_result.entities)
display(HTML(html))

print("\nLegend:")
print("  🟥 Pink: Names")
print("  🟩 Green: Dates")
print("  🟦 Blue: SSN")
print("  🟨 Yellow: Phone")
print("  🟧 Orange: Email")
print("  🟪 Purple: Address")

---
## 10. CLI Usage Examples

Command-line interface for PII operations.

In [ ]:
print("=" * 80)
print("CLI USAGE EXAMPLES")
print("=" * 80)
print("""
OpenMed provides a powerful CLI for PII detection and de-identification.

1. Extract PII from text:
   ```bash
   openmed pii extract \
     --text "Patient: John Doe, DOB: 01/15/1970" \
     --model openmed/OpenMed-PII-SuperClinical-Large-434M-v1 \
     --confidence-threshold 0.5
   ```

2. Extract PII from file:
   ```bash
   openmed pii extract \
     --input-file patient_note.txt \
     --output results.json
   ```

3. De-identify with mask method:
   ```bash
   openmed pii deidentify \
     --input-file patient_note.txt \
     --method mask \
     --output deidentified.txt
   ```

4. De-identify with date shifting:
   ```bash
   openmed pii deidentify \
     --text "Admission: 01/15/2025" \
     --method shift_dates \
     --date-shift-days 180
   ```

5. Batch processing:
   ```bash
   openmed pii batch-extract \
     --input-dir ./patient_notes/ \
     --output-dir ./results/ \
     --confidence-threshold 0.6
   ```

6. Interactive mode:
   ```bash
   openmed pii interactive
   ```

7. Get help:
   ```bash
   openmed pii --help
   openmed pii extract --help
   openmed pii deidentify --help
   ```
""")

print("\nTo see available CLI commands, run:")
print("  !openmed pii --help")

---
## Summary and Best Practices

### Key Takeaways

1. **Smart Merging (v0.5.0)** - Always enabled by default
   - Fixes fragmentation issues
   - Production-ready complete entities
   - Minimal performance overhead (~5-10%)

2. **De-identification Methods**
   - `mask`: Best for clinical review (maintains structure)
   - `remove`: Maximum privacy (minimal data)
   - `replace`: Research datasets (synthetic data)
   - `hash`: Linking records (deterministic)
   - `shift_dates`: Temporal analysis (preserves relationships)

3. **Confidence Thresholds**
   - 0.5-0.7: Recommended for most use cases
   - Lower: High recall (catch more PII)
   - Higher: High precision (fewer false positives)

4. **Batch Processing**
   - Use for multiple documents
   - Efficient resource usage
   - Progress tracking included

5. **HIPAA Compliance**
   - Covers all 18 identifier categories
   - Audit trails with mappings
   - Safe Harbor method supported

### Best Practices

✅ **DO:**
- Use `use_smart_merging=True` (default) for production
- Test with representative data
- Monitor entity quality (check for fragments)
- Keep mappings for audit trails
- Validate de-identified output

❌ **DON'T:**
- Disable smart merging without good reason
- Use very low thresholds without review
- Skip validation on production data
- Share mappings without encryption
- Assume 100% recall (always review edge cases)

### Performance Tips

- Use batch processing for multiple documents
- Adjust `batch_size` based on memory
- Cache model loading for repeated calls
- Monitor processing time with profiling

### Security Considerations

- Store mappings securely (encrypted)
- Limit access to original data
- Audit de-identification logs
- Follow organizational HIPAA policies
- Regular compliance reviews

---

## Resources

- **Documentation:** https://github.com/maziyarpanahi/openmed
- **Smart Merging Guide:** `docs/pii-smart-merging.md`
- **API Reference:** `docs/api-reference.md`
- **HIPAA Compliance:** `docs/hipaa-compliance.md`
- **Model Hub:** https://huggingface.co/openmed

---

**Version:** OpenMed v0.5.0+

**Last Updated:** 2026-01-13


In [36]:
# Re-run Evaluation for LLM with the new parsed entities
llm_model_pool = [(e.label, e.text) for e in llm_entities]
llm_matched, llm_missed = [], []

for exp_l, exp_t in expected_entities:
    found = False
    for i, (m_l, m_t) in enumerate(llm_model_pool):
        if labels_match(exp_l, m_l) and texts_match(exp_t, m_t):
            llm_matched.append((exp_l, exp_t, m_l, m_t))
            llm_model_pool.pop(i)
            found = True
            break
    if not found: llm_missed.append((exp_l, exp_t))

llm_tp, llm_fn, llm_fp = len(llm_matched), len(llm_missed), len(llm_model_pool)
llm_precision = llm_tp / (llm_tp + llm_fp) if (llm_tp + llm_fp) > 0 else 0
llm_recall = llm_tp / (llm_tp + llm_fn) if (llm_tp + llm_fn) > 0 else 0
llm_f1 = 2*llm_precision*llm_recall/(llm_precision+llm_recall) if (llm_precision+llm_recall) > 0 else 0

print("=" * 80)
print("  LLM (Llama-3.3-70B) PERFORMANCE REPORT")
print("=" * 80)
print(f"  Precision : {llm_precision:.2%}")
print(f"  Recall    : {llm_recall:.2%}")
print(f"  F1 Score  : {llm_f1:.2%}")

  LLM (Llama-3.3-70B) PERFORMANCE REPORT
  Precision : 89.29%
  Recall    : 94.94%
  F1 Score  : 92.02%


---
## 11. Per-Case Batch Evaluation

Split the teacher report into individual test cases, run them through `BatchProcessor`, and score each case independently plus produce an aggregate report.

In [25]:
import importlib, sys, os

# Reload in case the file changed during the session
if 'test_cases' in sys.modules:
    importlib.reload(sys.modules['test_cases'])

sys.path.insert(0, os.path.dirname(os.path.abspath('test_cases.py')))
from test_cases import TEST_CASES

print(f'Loaded {len(TEST_CASES)} test cases')
for tc in TEST_CASES:
    print(f"  [{tc['id']:35s}]  {len(tc['expected']):3d} expected  |  ~{len(tc['text'].split()):4d} words")


Loaded 12 test cases
  [header_superintendent              ]   93 expected  |  ~1821 words
  [teacher_1_thornton                 ]   59 expected  |  ~1681 words
  [teacher_2_subramaniam              ]   45 expected  |  ~1644 words
  [teacher_3_menendez                 ]   33 expected  |  ~1572 words
  [teacher_4_castillo                 ]   58 expected  |  ~1655 words
  [teacher_5_okafor                   ]   42 expected  |  ~1583 words
  [teacher_6_alrashid                 ]   42 expected  |  ~1583 words
  [teacher_7_kowalski                 ]   40 expected  |  ~1606 words
  [teacher_8_nguyen                   ]   36 expected  |  ~1589 words
  [teacher_9_romano                   ]   38 expected  |  ~1612 words
  [teacher_10_tanaka                  ]   51 expected  |  ~1616 words
  [district_hr_notes                  ]  109 expected  |  ~1813 words


In [26]:
import time
from openmed import ModelLoader

MODEL_NAME = 'openmed/OpenMed-PII-SuperClinical-Large-434M-v1'

load_start = time.perf_counter()
batch_loader = ModelLoader()
batch_loader.create_pipeline(MODEL_NAME, task="token-classification", aggregation_strategy="simple")
load_ms = (time.perf_counter() - load_start) * 1000
print(f"Model loaded in {load_ms:.1f}ms — run the next cell to batch-evaluate all test cases.")


Loading weights: 100%|██████████| 392/392 [00:00<00:00, 8469.94it/s]


Model loaded in 3287.9ms — run the next cell to batch-evaluate all test cases.


In [27]:
import time
from openmed import BatchProcessor

# ── Helpers ───────────────────────────────────────────────────────────────────
DOB_AS_DATE_ACCEPTABLE = True
HIGH_VALUE  = {"ssn", "date_of_birth", "phone_number", "email", "street_address"}
BENIGN_FP   = {"date", "education_level", "url", "company_name", "country", "pin", "bank_routing_number"}

def _norm(t):
    return t.strip().lower().replace("\n", " ").replace("  ", " ")

def _lmatch(el, ml):
    return el == ml or (DOB_AS_DATE_ACCEPTABLE and el == "date_of_birth" and ml == "date")

def _tmatch(et, mt):
    e, m = _norm(et), _norm(mt)
    return e == m or e in m or m in e

def score_case(expected, entities):
    pool = [(e.label, e.text) for e in entities]
    matched, missed = [], []
    for el, et in expected:
        hit = False
        for i, (ml, mt) in enumerate(pool):
            if _lmatch(el, ml) and _tmatch(et, mt):
                matched.append((el, et, ml, mt)); pool.pop(i); hit = True; break
        if not hit:
            missed.append((el, et))
    fp_crit   = [(l, t) for l, t in pool if l not in BENIGN_FP]
    fp_benign = [(l, t) for l, t in pool if l in BENIGN_FP]
    return matched, missed, fp_crit, fp_benign

# ── Batch run (reuses pre-loaded model) ───────────────────────────────────────
processor = BatchProcessor(
    model_name=MODEL_NAME,
    confidence_threshold=0.5,
    group_entities=True,
    continue_on_error=True,
    loader=batch_loader,
)

t0 = time.perf_counter()
batch_result = processor.process_texts([tc["text"] for tc in TEST_CASES])
total_s = time.perf_counter() - t0

# ── Per-case scoring ──────────────────────────────────────────────────────────
W = 88
agg = {"tp": 0, "fn": 0, "fp": 0}
case_metrics = []

print("=" * W)
print("  PER-CASE BATCH EVALUATION REPORT")
print(f"  Model : {MODEL_NAME}")
print(f"  Model load : {load_ms/1000:.2f}s  |  Batch inference : {total_s:.2f}s")
print(f"  Cases: {batch_result.total_items}  |  Success: {batch_result.successful_items}  |  Failed: {batch_result.failed_items}")
print("=" * W)

for tc, item in zip(TEST_CASES, batch_result.items):
    print(f"\n{'─' * W}")
    print(f"  CASE: {tc['id']}")
    print(f"{'─' * W}")

    if not item.success:
        print(f"  ERROR: {item.error}")
        case_metrics.append({"id": tc["id"], "error": item.error})
        continue

    matched, missed, fp_crit, fp_benign = score_case(tc["expected"], item.result.entities)
    tp, fn, fp = len(matched), len(missed), len(fp_crit) + len(fp_benign)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = 2*prec*rec / (prec+rec) if (prec+rec) > 0 else 0.0

    agg["tp"] += tp; agg["fn"] += fn; agg["fp"] += fp
    case_metrics.append({"id": tc["id"], "tp": tp, "fn": fn, "fp": fp,
                         "precision": prec, "recall": rec, "f1": f1})

    print(f"  Detected: {len(item.result.entities)}  |  Expected: {len(tc['expected'])}")
    print(f"  TP={tp}  FN={fn}  FP={fp}  |  Precision={prec:.2%}  Recall={rec:.2%}  F1={f1:.2%}")

    hv = [(l,t) for l,t in missed if l in HIGH_VALUE]
    ot = [(l,t) for l,t in missed if l not in HIGH_VALUE]
    if hv:
        print(f"\n  Missed HIGH-VALUE ({len(hv)}):")
        for l, t in hv: print(f"    [{l:15s}] '{t}'")
    if ot:
        print(f"  Missed other ({len(ot)}):")
        for l, t in ot: print(f"    [{l:15s}] '{t}'")
    if fp_crit:
        print(f"\n  Critical FP ({len(fp_crit)}):")
        for l, t in fp_crit: print(f"    [{l:15s}] '{t.strip()}'")
    if fp_benign:
        print(f"  Benign FP ({len(fp_benign)}):")
        for l, t in fp_benign: print(f"    [{l:15s}] '{t.strip()}'")

# ── Aggregate table ───────────────────────────────────────────────────────────
atp, afn, afp = agg["tp"], agg["fn"], agg["fp"]
ap = atp/(atp+afp) if (atp+afp) > 0 else 0.0
ar = atp/(atp+afn) if (atp+afn) > 0 else 0.0
af = 2*ap*ar/(ap+ar) if (ap+ar) > 0 else 0.0

print(f"\n{'=' * W}")
print("  AGGREGATE SUMMARY")
print(f"{'=' * W}")
print(f"  {'Case':<35}  {'Precision':>10}  {'Recall':>8}  {'F1':>8}  {'TP':>5}  {'FN':>5}  {'FP':>5}")
print(f"  {'-'*35}  {'-'*10}  {'-'*8}  {'-'*8}  {'-'*5}  {'-'*5}  {'-'*5}")
for m in case_metrics:
    if "error" in m:
        print(f"  {m['id']:<35}  {'ERROR':>10}")
    else:
        print(f"  {m['id']:<35}  {m['precision']:>10.2%}  {m['recall']:>8.2%}  "
              f"{m['f1']:>8.2%}  {m['tp']:>5}  {m['fn']:>5}  {m['fp']:>5}")
print(f"  {'─'*35}  {'─'*10}  {'─'*8}  {'─'*8}  {'─'*5}  {'─'*5}  {'─'*5}")
print(f"  {'OVERALL':<35}  {ap:>10.2%}  {ar:>8.2%}  {af:>8.2%}  {atp:>5}  {afn:>5}  {afp:>5}")
print("=" * W)


  PER-CASE BATCH EVALUATION REPORT
  Model : openmed/OpenMed-PII-SuperClinical-Large-434M-v1
  Model load : 3.29s  |  Batch inference : 566.80s
  Cases: 12  |  Success: 12  |  Failed: 0

────────────────────────────────────────────────────────────────────────────────────────
  CASE: header_superintendent
────────────────────────────────────────────────────────────────────────────────────────
  Detected: 674  |  Expected: 93
  TP=93  FN=0  FP=581  |  Precision=13.80%  Recall=100.00%  F1=24.25%

  Critical FP (470):
    [occupation     ] 'Superintendent'
    [phone_number   ] '(217) 555-0192'
    [street_address ] '88 District Boulevard'
    [postcode       ] '62704'
    [last_name      ] 'Danielson'
    [street_address ] '88 District Boulevard'
    [state          ] 'IL'
    [postcode       ] '62704'
    [first_name     ] 'Margaret'
    [last_name      ] 'Harrington'
    [phone_number   ] '(217) 555-0192'
    [email          ] 'm.harrington@greenfieldusd.edu'
    [occupation     ] 'HR D